In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import librosa
import numpy as np
import pandas as pd
import os
import ssl
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ssl._create_default_https_context = ssl._create_unverified_context

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [2]:
N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 41

In [4]:
clip_df = pd.read_csv('../clips_metadata.csv')
print(f'Total clips: {len(clip_df)}')
print(f'Species: {clip_df["species"].nunique()}')

le = LabelEncoder()
clip_df['label'] = le.fit_transform(clip_df['species'])
print(f'Classes: {len(le.classes_)}')

Total clips: 49682
Species: 41
Classes: 41


In [5]:
class AviaNetDataset(Dataset):
    def __init__(self, df, augment_data=False):
        self.df = df.reset_index(drop=True)
        self.augment_data = augment_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        try:
            y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
        except Exception:
            y = np.zeros(SR * DURATION)
            sr = SR

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

    def augment(self, y, sr):
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
        shift = np.random.randint(0, max(1, sr // 4))
        y = np.roll(y, shift)
        return y

print('Dataset class defined')

Dataset class defined
